## MF Methods

In [ ]:
!pip install nimfa scikit-learn pandas numpy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 30.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

# Load expression matrix
# expr = pd.read_csv("dropSeq.csv", index_col=0)
expr = pd.read_csv("dream5_net2_expression_data.tsv", sep="\t")

# Convert to numpy array
X = expr.values.astype(float)

print(X.shape)  # (n_cells, n_genes)

(160, 2810)


In [ ]:
def explained_variance(X, X_hat):
    return 1 - (np.linalg.norm(X - X_hat) ** 2 /
                np.linalg.norm(X) ** 2)

In [ ]:
import nimfa

methods = {
    "Bd": nimfa.Bd,
    "Bmf": nimfa.Bmf,
    "Icm": nimfa.Icm,
    "Lfnmf": nimfa.Lfnmf,
    "Lsnmf": nimfa.Lsnmf,
    "Nmf": nimfa.Nmf,
    "Nsnmf": nimfa.Nsnmf,
    "Pmf": nimfa.Pmf,
    "Psmf": nimfa.Psmf,
    "Snmf": nimfa.Snmf,
    "Snmnmf": nimfa.Snmnmf,
    "Pmfcc": nimfa.Pmfcc,
    "SepNmf": nimfa.SepNmf
}

In [ ]:
ranks = [2, 5, 10, 20, 30, 50]

In [ ]:
import numpy as np
import os
import pandas as pd
from datetime import datetime

# NumPy compatibility fix
if not hasattr(np, "mat"):
    np.mat = np.asmatrix

def explained_variance(X, X_hat):
    return 1 - (np.linalg.norm(X - X_hat) ** 2 /
                np.linalg.norm(X) ** 2)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
txt_out = f"nimfa_all_methods_variance_{timestamp}.txt"

with open(txt_out, "w") as f:
    f.write("Method\tRank\tExplainedVariance\tStatus\n")

results = []

for method_name, method_class in methods.items():
    for r in ranks:
        print(f"Running {method_name} | rank={r}")

        try:
            model = method_class(
                X,
                rank=r,
                max_iter=200,
                seed="random_vcol"
            )

            fit = model()

            # Some methods expose basis/coef slightly differently
            W = np.array(fit.basis())
            H = np.array(fit.coef())

            X_hat = W @ H
            var_exp = explained_variance(X, X_hat)

            results.append({
                "Method": method_name,
                "Rank": r,
                "ExplainedVariance": var_exp,
                "Status": "OK"
            })

            with open(txt_out, "a") as f:
                f.write(f"{method_name}\t{r}\t{var_exp:.6f}\tOK\n")
                f.flush()
                os.fsync(f.fileno())

        except Exception as e:
            # Log failure but continue
            results.append({
                "Method": method_name,
                "Rank": r,
                "ExplainedVariance": np.nan,
                "Status": "FAILED"
            })

            with open(txt_out, "a") as f:
                f.write(f"{method_name}\t{r}\tNA\tFAILED: {str(e)}\n")
                f.flush()
                os.fsync(f.fileno())

            print(f"⚠️ Failed: {method_name}, rank={r}")


results_df = pd.DataFrame(results)
results_df.to_csv(f"nimfa_all_methods_variance_{timestamp}.csv")

Running Bd | rank=2
Running Bd | rank=5
Running Bd | rank=10
Running Bd | rank=20
Running Bd | rank=30
Running Bd | rank=50
Running Bmf | rank=2
Running Bmf | rank=5
Running Bmf | rank=10
Running Bmf | rank=20
Running Bmf | rank=30
Running Bmf | rank=50
Running Icm | rank=2
Running Icm | rank=5
Running Icm | rank=10
Running Icm | rank=20
Running Icm | rank=30
Running Icm | rank=50
Running Lfnmf | rank=2
Running Lfnmf | rank=5
Running Lfnmf | rank=10


## Linear Methods

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA, TruncatedSVD

In [ ]:
# X shape: (n_samples, n_features)
X = X.astype(float)

# Optional but recommended: log-normalisation
X_log = np.log1p(X)

In [ ]:
def explained_variance_recon(X, X_hat):
    return 1 - (np.linalg.norm(X - X_hat) ** 2 /
                np.linalg.norm(X) ** 2)

In [ ]:
pca_results = []

for r in ranks:
    print(f"Running PCA with n_components={r}")

    pca = PCA(n_components=r, svd_solver="auto", random_state=0)
    X_pca = pca.fit_transform(X_log)

    # Reconstruct
    X_hat = pca.inverse_transform(X_pca)

    var_exp = explained_variance_recon(X_log, X_hat)

    pca_results.append({
        "Method": "PCA",
        "Rank": r,
        "ExplainedVariance": var_exp
    })

Running PCA with n_components=2
Running PCA with n_components=5
Running PCA with n_components=10
Running PCA with n_components=20
Running PCA with n_components=30
Running PCA with n_components=50


In [ ]:
svd_results = []

for r in ranks:
    print(f"Running SVD with n_components={r}")

    svd = TruncatedSVD(n_components=r, random_state=0)
    X_svd = svd.fit_transform(X_log)

    # Reconstruction
    X_hat = np.dot(X_svd, svd.components_)

    var_exp = explained_variance_recon(X_log, X_hat)

    svd_results.append({
        "Method": "SVD",
        "Rank": r,
        "ExplainedVariance": var_exp
    })

Running SVD with n_components=2
Running SVD with n_components=5
Running SVD with n_components=10
Running SVD with n_components=20
Running SVD with n_components=30
Running SVD with n_components=50


In [ ]:
linear_results_df = pd.DataFrame(pca_results + svd_results)
linear_results_df
linear_results = pd.DataFrame(linear_results_df)
linear_results.to_csv(f"linear_methods_variance.csv")